In [1]:
import qewton
import math 

In [2]:
T = qewton.Variable("t", dim=1)
U = qewton.Variable("u", dim=1)

In [3]:
interval = qewton.geometries.Interval(T, 0, 2)

inner_sampler = qewton.RandomUniformSampler(
        interval, n_points=qewton.DiscreteHyperparameter((100, 5000), name="Inner Sampler Points")
    )
left_sampler = qewton.GridSampler(interval.boundary_left, 1)

In [ ]:
model = qewton.FCN(
    in_neurons=T,
    hidden_neurons=qewton.DiscreteHyperparameter((4, 64)),
    out_neurons=U,
    n_hidden_layers=qewton.DiscreteHyperparameter((1, 5)),
    activation=qewton.bb.Tanh,
)

In [5]:
cos = qewton.bb.Cos()

def residual_fun(u: U, t: T):  # type: ignore
    return u.gradient(t) - (4.0*math.pi*cos(4.0*math.pi*t) + 2.0*t)

constraint = qewton.PINNConstraint(residual_fun, name="PINNConstraint")
ode_graph = qewton.PINNPipeline(inner_sampler, [model], constraint)

In [6]:
def boundary_residual_fun(u: U):  # type: ignore
    return u

boundary_constraint = qewton.constraints.PINNConstraint(
    boundary_residual_fun, 
    name="InitialConstraint", 
    weight=qewton.ContinuousHyperparameter((1, 100), name="Boundary Weight")
)
boundary_graph = qewton.PINNPipeline(left_sampler, [model], boundary_constraint)

In [7]:
validation_constraint = qewton.MSEConstraint(name="ValidationConstraint", 
                                             evaluated_in_mode=qewton.EvaluationPhase.VALIDATION)
validation_pipeline = qewton.Graph()
Sin = qewton.bb.Sin()

def real_solution(t: T):  # type: ignore
    return Sin(4.0*math.pi*t) + t**2

with validation_pipeline.tracker(0):
    t = inner_sampler()
    model_out = model(t)
    validation_constraint(model_out, real_solution(t))

In [8]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.001,
    max_iterations=1000,
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=adam_phase,
    graphs=[boundary_graph, ode_graph, validation_pipeline],
    training_objectives=[boundary_constraint, constraint],
    callbacks=[
        qewton.optim.ModelParameterSizeCallback([model])
    ],
    device="cuda:0",
)


In [9]:
save_path = "results"   

tuner = qewton.optim.tuner.GridSearchTuner(
    trainer,
    tuning_objectives=[validation_constraint],
    trial_number=40,
    devices=[
        "cuda:0",
        "cuda:1",
        "cuda:2",
        "cuda:3",
    ],
    save_path=save_path,
    trials_per_device=1,
)
tuner.run()

--- Start Tuning ---
Working on trials 0 - 10


/home/tomfre/miniconda3/envs/PIONEER/lib/python3.10/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/tomfre/miniconda3/envs/PIONEER/lib/python3.10/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/tomfre/miniconda3/envs/PIONEER/lib/python3.10/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/tomfre/miniconda3/envs/PIONEER/lib/python3.10/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/tomfre/miniconda3/envs/PIONEER/lib/python3.10/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


Working on trials 10 - 20
Working on trials 20 - 30
Working on trials 30 - 40
Working on trials 40 - 40
--- Cleaning up ---


[W1013 17:40:03.585875243 CudaIPCTypes.cpp:16] Producer process has been terminated before all shared CUDA tensors released. See Note [Sharing CUDA tensors]


--- Finished Tuning ---


In [10]:
tuning_analyer = qewton.optim.TuningAnalyzer(save_path)

In [11]:
tuning_analyer.best_run()

Boundary Weight                       1.0
Inner Sampler Points                 5000
Hidden Neurons                         34
Hidden Layer                            5
ValidationConstraint             0.000004
Termination Reason      Training finished
Training Time [s]                       4
Save Path                             NaN
Parameter size [MB]                  0.02
Name: 9, dtype: object

In [12]:
tuning_analyer.param_importance()

Hidden Neurons          0.423409
Boundary Weight         0.215505
Hidden Layer            0.078613
Inner Sampler Points    0.001604
dtype: float64

In [13]:
fig = tuning_analyer.parallel_coordinates_plot(
    axes=["Hidden Neurons", "Hidden Layer", "Inner Sampler Points", "Boundary Weight", "Parameter size [MB]"],
    objectives=["ValidationConstraint"],
    log_axes=["ValidationConstraint"]
    )
fig.show()